# ArcRho Excel UserForm Preview

Use this notebook while Excel is already open with the ArcRho add-in or VBA project loaded. It connects to the current Excel instance and calls individual UserForm launch macros so you can preview layout on different displays and Windows scaling settings.

Recommended test flow: open Excel on the target monitor, open a workbook that has the add-in loaded, run the setup cell, then run one preview cell at a time. Close each form before opening another, especially modal forms.

In [1]:
import time
from dataclasses import dataclass
from pathlib import Path
import stat
import subprocess
import win32com.client as win32

REPO_ROOT = Path(r"E:\\XWSpace\\Repos\\ArcRho")
TEST_WORKBOOK_PATH = REPO_ROOT / "excel-addin" / "ui-test" / "TestUserForms.xlsx"
ADDIN_PATH = Path(r"E:\\ArcRho Server\\Excel Add-ins\\ArcRho.xlam")
BETA_ADDIN_PATH = REPO_ROOT / "excel-addin" / "beta" / "ARCRHO_BETA.xlam"
BUILD_SCRIPT_PATH = REPO_ROOT / "excel-addin" / "tools" / "build_xlam.ps1"
RELEASE_SCRIPT_PATH = REPO_ROOT / "excel-addin" / "tools" / "release_xlam.ps1"
for required_path in (TEST_WORKBOOK_PATH, ADDIN_PATH, BETA_ADDIN_PATH, BUILD_SCRIPT_PATH, RELEASE_SCRIPT_PATH):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

try:
    excel = win32.GetActiveObject("Excel.Application")
except Exception:
    excel = win32.Dispatch("Excel.Application")
excel.Visible = True

test_wb = None
for wb in excel.Workbooks:
    try:
        if Path(wb.FullName).resolve().samefile(TEST_WORKBOOK_PATH):
            test_wb = wb
            break
    except Exception:
        pass

if test_wb is None:
    test_wb = excel.Workbooks.Open(str(TEST_WORKBOOK_PATH))

test_wb.Activate()
active_wb = test_wb

print(f"Excel version: {excel.Version}")
print(f"Test workbook: {active_wb.FullName if active_wb else '(none)'}")
print("Open workbooks:")
for wb in excel.Workbooks:
    print(f"- {wb.Name}")

Excel version: 16.0
Test workbook: E:\XWSpace\Repos\ArcRho\excel-addin\ui-test\TestUserForms.xlsx
Open workbooks:
- TestUserForms.xlsx


## Helpers

`Application.Run` can call parameterless VBA macros directly. Ribbon callbacks expect an `IRibbonControl`; for preview testing, this notebook passes `None`, which works in many local callback cases because the callback body does not use the control argument.

In [2]:
@dataclass(frozen=True)
class FormPreview:
    form_name: str
    macro_name: str
    needs_ribbon_arg: bool = False
    note: str = ""


FORM_PREVIEWS = {
    "progress": FormPreview("ufProgressBar", "Show_ufProgressBar"),
    "loading": FormPreview("ufLoading", "Show_ufLoading"),
    "reserving_classes": FormPreview("ufLoadReservingClasses", "uiLoadReservingClasses2", True),
    "select_dataset": FormPreview("ufSelectDataset", "uiSelectDatasets", True),
    "build_tri": FormPreview("ufBuildTri", "uiCheckUpdates", True, "Modal form; close it before continuing."),
    "settings": FormPreview("ufSettings", "uiSettings", True),
    "about": FormPreview("ufAbout", "uiAbout", True),
}


def _macro_candidates(macro_name):
    names = [macro_name]
    for wb in excel.Workbooks:
        names.append(f"'{wb.Name}'!{macro_name}")
    return names


def run_macro(macro_name, *args):
    last_error = None
    for candidate in _macro_candidates(macro_name):
        try:
            return excel.Run(candidate, *args)
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Unable to run macro '{macro_name}'. Last Excel error: {last_error}")


def preview_form(key):
    item = FORM_PREVIEWS[key]
    print(f"Opening {item.form_name} via {item.macro_name} ...")
    if item.note:
        print(item.note)
    args = (None,) if item.needs_ribbon_arg else ()
    try:
        result = run_macro(item.macro_name, *args)
    except Exception as exc:
        if item.needs_ribbon_arg:
            print("This ribbon callback may not accept a Python None argument in your Excel session.")
            print("Add or run a parameterless VBA wrapper such as:")
            print(f"Public Sub Preview_{item.form_name}(): {item.form_name}.Show vbModeless: End Sub")
        raise exc
    time.sleep(0.2)
    return result


def preview_progress_in_active_workbook():
    active_workbook = excel.ActiveWorkbook
    if active_workbook is None:
        raise RuntimeError("Excel has no active workbook.")
    active_workbook.Activate()
    print(f"Opening ufProgressBar in active workbook context: {active_workbook.Name}")
    return preview_form("progress")


def _same_file(candidate_path, target_path):
    try:
        return Path(candidate_path).resolve().samefile(target_path)
    except Exception:
        return False


def _find_excel_addin(addin_path):
    for candidate in excel.AddIns:
        if _same_file(candidate.FullName, addin_path):
            return candidate
    return None


def load_excel_addin(addin_path):
    addin_path = Path(addin_path)
    addin = _find_excel_addin(addin_path)
    if addin is None:
        addin = excel.AddIns.Add(str(addin_path), False)
    addin.Installed = True
    test_wb.Activate()
    print(f"Loaded add-in: {addin.FullName}")
    return addin


def unload_excel_addin(addin_path):
    addin_path = Path(addin_path)
    unloaded = False
    addin = _find_excel_addin(addin_path)
    if addin is not None:
        addin.Installed = False
        unloaded = True

    for wb in list(excel.Workbooks):
        if _same_file(wb.FullName, addin_path):
            wb.Close(SaveChanges=False)
            unloaded = True

    test_wb.Activate()
    print(f"Unloaded add-in: {addin_path}" if unloaded else f"Add-in was not loaded: {addin_path}")
    return unloaded


def load_arcrho_addin():
    return load_excel_addin(ADDIN_PATH)


def unload_arcrho_addin():
    return unload_excel_addin(ADDIN_PATH)


def load_beta_addin():
    return load_excel_addin(BETA_ADDIN_PATH)


def unload_beta_addin():
    return unload_excel_addin(BETA_ADDIN_PATH)


def _run_powershell_script(script_path, timeout_seconds=300):
    result = subprocess.run(
        [
            "powershell",
            "-NoProfile",
            "-ExecutionPolicy",
            "Bypass",
            "-File",
            str(script_path),
        ],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
        timeout=timeout_seconds,
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"Script failed with exit code {result.returncode}: {script_path}")
    return result


def _clear_readonly(path):
    path = Path(path)
    try:
        path.chmod(path.stat().st_mode | stat.S_IWRITE)
    except Exception:
        pass


def _touch_userform_designer(component):
    window = None
    touched = False
    try:
        window = component.Activate()
        if window is not None:
            window.Visible = True
        time.sleep(0.15)
    except Exception:
        pass

    try:
        designer = component.Designer
        for attr in ("Width", "Height", "InsideWidth", "InsideHeight"):
            try:
                value = getattr(designer, attr)
                setattr(designer, attr, value + 0.75)
                setattr(designer, attr, value)
                touched = True
                break
            except Exception:
                pass

        if not touched:
            controls = designer.Controls
            for idx in range(min(int(controls.Count), 3)):
                control = controls.Item(idx)
                try:
                    value = control.Width
                    control.Width = value + 0.75
                    control.Width = value
                    touched = True
                    break
                except Exception:
                    pass
    except Exception:
        pass

    try:
        if window is not None:
            window.Close()
    except Exception:
        pass

    return touched


def normalize_userform_layouts_in_addin(addin_path=BETA_ADDIN_PATH):
    addin_path = Path(addin_path)
    unload_excel_addin(addin_path)
    _clear_readonly(addin_path)

    for wb in list(excel.Workbooks):
        try:
            if Path(wb.FullName).resolve().samefile(addin_path):
                wb.Close(SaveChanges=False)
        except Exception:
            pass

    print(f"Opening add-in for visible UserForm normalization: {addin_path}")
    workbook = excel.Workbooks.Open(str(addin_path), UpdateLinks=False, ReadOnly=False)
    old_is_addin = bool(workbook.IsAddin)
    try:
        excel.Visible = True
        try:
            excel.VBE.MainWindow.Visible = True
        except Exception:
            pass

        touched = []
        missed = []
        for component in workbook.VBProject.VBComponents:
            if int(component.Type) == 3:
                if _touch_userform_designer(component):
                    touched.append(component.Name)
                else:
                    missed.append(component.Name)

        workbook.IsAddin = old_is_addin
        workbook.Save()
        print("Normalized UserForms: " + (", ".join(touched) if touched else "(none)"))
        if missed:
            print("Could not touch UserForms: " + ", ".join(missed))
        return touched
    finally:
        workbook.Close(SaveChanges=False)
        test_wb.Activate()


def rebuild_arcrho_addin(timeout_seconds=300):
    unload_arcrho_addin()
    unload_beta_addin()
    print(f"Building beta add-in from source: {BETA_ADDIN_PATH}")
    _run_powershell_script(BUILD_SCRIPT_PATH, timeout_seconds)
    print(f"Releasing beta add-in to installed add-in: {ADDIN_PATH}")
    _run_powershell_script(RELEASE_SCRIPT_PATH, timeout_seconds)
    if not ADDIN_PATH.exists():
        raise FileNotFoundError(ADDIN_PATH)
    print(f"Rebuilt installed add-in: {ADDIN_PATH}")
    return ADDIN_PATH


def rebuild_load_arcrho_and_preview(form_key="progress"):
    rebuild_arcrho_addin()
    load_arcrho_addin()
    return preview_form(form_key)


def rebuild_load_beta_and_preview(form_key="progress"):
    return rebuild_load_arcrho_and_preview(form_key)

print("Available previews:")
for key, item in FORM_PREVIEWS.items():
    print(f"- {key}: {item.form_name}")

Available previews:
- progress: ufProgressBar
- loading: ufLoading
- reserving_classes: ufLoadReservingClasses
- select_dataset: ufSelectDataset
- build_tri: ufBuildTri
- settings: ufSettings
- about: ufAbout


## Tools

Run these cells to load/unload the installed add-in or rebuild the installed add-in and preview a form.

In [ ]:
preview_progress_in_active_workbook()

In [14]:
load_arcrho_addin()

Loaded add-in: E:\ArcRho Server\Excel Add-ins\ArcRho.xlam


<COMObject <unknown>>

In [13]:
unload_arcrho_addin()

Unloaded add-in: E:\ArcRho Server\Excel Add-ins\ArcRho.xlam


True

In [31]:
rebuild_load_arcrho_and_preview("progress")

Unloaded add-in: E:\ArcRho Server\Excel Add-ins\ArcRho.xlam
Unloaded add-in: E:\XWSpace\Repos\ArcRho\excel-addin\beta\ARCRHO_BETA.xlam
Building beta add-in from source: E:\XWSpace\Repos\ArcRho\excel-addin\beta\ARCRHO_BETA.xlam
Using ribbon tab label: ARCRHO_BETA
Updated ribbon XML: E:\XWSpace\Repos\ArcRho\excel-addin\tools\customUI.xml
Updated workbook title: ARCRHO_BETA
Importing Core.bas as Core
Importing ExportCode.bas as ExportCode
Importing Helper.bas as Helper
Importing JsonParser.bas as JsonParser
Importing mod_LoadReservingClasses.bas as mod_LoadReservingClasses
Importing mod_PendingWatcher.bas as mod_PendingWatcher
Importing mod_SelectDataset.bas as mod_SelectDataset
Importing mod_Test.bas as mod_Test
Importing Register_Descriptions.bas as Register_Descriptions
Importing Ribbon_Control.bas as Ribbon_Control
Importing Ribbon_Functions.bas as Ribbon_Functions
Importing Show_UserForms.bas as Show_UserForms
Importing UDF_ArcRho.bas as UDF_ArcRho
Importing UDF_TRIMAVG.bas as UDF_TR

## Preview One Form

Change `form_key` and run the cell. Available keys: `progress`, `loading`, `reserving_classes`, `select_dataset`, `build_tri`, `settings`, `about`.

In [ ]:
form_key = "progress"
preview_form(form_key)

## Quick Buttons

Run any single line below to preview that form.

In [ ]:
preview_form("progress")
# preview_form("loading")
# preview_form("reserving_classes")
# preview_form("select_dataset")
# preview_form("build_tri")
# preview_form("settings")
# preview_form("about")

## DPI / Display Notes

For consistent screenshots, close and reopen Excel after changing Windows display scale. Test with Excel started on each target monitor: 4K at 125%, 1080p at 100%, and 1080p at 125%. UserForms are most reliable when launched fresh on the monitor being tested.